# PostgreSQL Data Warehouse ETL

## Overview

This notebook implements the ETL process that loads the cleaned ChriLy datasets into PostgreSQL and builds a dimensional data warehouse.

The pipeline consists of four stages:

1. Load the cleaned datasets from the local project.
2. Import the datasets into a PostgreSQL staging schema.
3. Execute SQL scripts to create and populate the dimensional data warehouse.
4. Validate the resulting warehouse before connecting it to Power BI.

The staging layer acts as a temporary landing zone for the cleaned datasets, while the warehouse layer is optimized for analytical queries using a star schema.

In [13]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from sqlalchemy import text

In [14]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"
CLEAN_PATH = DATA_PATH / "clean"

SQL_PATH = PROJECT_ROOT / "sql"

In [16]:
DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/"
    f"{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("✓ SQLAlchemy engine created.")

✓ SQLAlchemy engine created.


In [9]:
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar()

print(version)

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit


In [6]:
customers = pd.read_csv(CLEAN_PATH / "chrily_customers.csv")
orders = pd.read_csv(CLEAN_PATH / "chrily_orders.csv")
order_items = pd.read_csv(CLEAN_PATH / "chrily_order_items.csv")
order_payments = pd.read_csv(CLEAN_PATH / "chrily_order_payments.csv")
order_reviews = pd.read_csv(CLEAN_PATH / "chrily_order_reviews.csv")
products = pd.read_csv(CLEAN_PATH / "chrily_products.csv")
sellers = pd.read_csv(CLEAN_PATH / "chrily_sellers.csv")
geolocation = pd.read_csv(CLEAN_PATH / "chrily_geolocation.csv")
category_translation = pd.read_csv(CLEAN_PATH / "chrily_category_translation.csv")

print("✓ All cleaned datasets loaded.")

✓ All cleaned datasets loaded.


In [27]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["purchase_date"] = pd.to_datetime(
    orders["purchase_date"], errors="coerce"
).dt.date


## Create PostgreSQL Schemas

Two schemas are used throughout the ETL pipeline:

- **staging**: Temporary landing area for the cleaned datasets.
- **chrily**: Dimensional data warehouse containing fact and dimension tables.

In [7]:
with engine.begin() as conn:

    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS staging;
    """))

    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS chrily;
    """))

print("✓ Schemas created successfully.")

✓ Schemas created successfully.


## Load Cleaned Datasets into the Staging Layer

In [8]:
datasets = {
    "stg_customers": customers,
    "stg_orders": orders,
    "stg_order_items": order_items,
    "stg_order_payments": order_payments,
    "stg_order_reviews": order_reviews,
    "stg_products": products,
    "stg_sellers": sellers,
    "stg_geolocation": geolocation,
    "stg_category_translation": category_translation
}

In [9]:
for table_name, df in datasets.items():

    df.to_sql(
        name=table_name,
        con=engine,
        schema="staging",
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=5000
    )

    print(f"✓ {table_name:<30} {len(df):>8,} rows")

✓ stg_customers                    99,441 rows
✓ stg_orders                       99,441 rows
✓ stg_order_items                 112,650 rows
✓ stg_order_payments              103,875 rows
✓ stg_order_reviews                99,224 rows
✓ stg_products                     32,951 rows
✓ stg_sellers                       3,095 rows
✓ stg_geolocation                1,000,163 rows
✓ stg_category_translation             71 rows


In [28]:
orders.to_sql(
    name="stg_orders",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

print("✓ stg_orders reloaded")

✓ stg_orders reloaded


## Validate the Staging Layer

In [29]:
query = """
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'staging'
ORDER BY table_name;
"""

pd.read_sql(query, engine)

,table_name
0,stg_category_translation
1,stg_customers
2,stg_geolocation
3,stg_order_items
4,stg_order_payments
5,stg_order_reviews
6,stg_orders
7,stg_products
8,stg_sellers


## Create the Data Warehouse Structure

The warehouse schema is created by executing SQL scripts stored in the project's `sql` directory.

In [17]:
def execute_sql_file(sql_file):

    with open(sql_file, "r", encoding="utf-8") as f:
        sql = f.read()

    with engine.begin() as conn:
        conn.execute(text(sql))

    print(f"✓ {sql_file.name} executed")

In [34]:
execute_sql_file(
    SQL_PATH / "02_create_dimensions.sql"
)

✓ 02_create_dimensions.sql executed


In [35]:
execute_sql_file(
    SQL_PATH / "03_create_fact_table.sql"
)

✓ 03_create_fact_table.sql executed


In [37]:
execute_sql_file(
    SQL_PATH / "04_build_warehouse.sql"
)

✓ 04_build_warehouse.sql executed


In [38]:
validation_queries = {
    "dim_customer": "SELECT COUNT(*) AS rows FROM chrily.dim_customer;",
    "dim_seller": "SELECT COUNT(*) AS rows FROM chrily.dim_seller;",
    "dim_product": "SELECT COUNT(*) AS rows FROM chrily.dim_product;",
    "dim_date": "SELECT COUNT(*) AS rows FROM chrily.dim_date;",
    "fact_orders": "SELECT COUNT(*) AS rows FROM chrily.fact_orders;"
}

for table, query in validation_queries.items():
    print("=" * 60)
    print(table.upper())
    print("=" * 60)
    display(pd.read_sql(query, engine))

DIM_CUSTOMER


,rows
0,99441


DIM_SELLER


,rows
0,3095


DIM_PRODUCT


,rows
0,32951


DIM_DATE


,rows
0,634


FACT_ORDERS


,rows
0,112644


In [39]:
checks = {

    "Missing Customers": """
        SELECT COUNT(*) AS missing
        FROM chrily.fact_orders f
        LEFT JOIN chrily.dim_customer d
        ON f.customer_key = d.customer_key
        WHERE d.customer_key IS NULL;
    """,

    "Missing Products": """
        SELECT COUNT(*) AS missing
        FROM chrily.fact_orders f
        LEFT JOIN chrily.dim_product d
        ON f.product_key = d.product_key
        WHERE d.product_key IS NULL;
    """,

    "Missing Sellers": """
        SELECT COUNT(*) AS missing
        FROM chrily.fact_orders f
        LEFT JOIN chrily.dim_seller d
        ON f.seller_key = d.seller_key
        WHERE d.seller_key IS NULL;
    """,

    "Missing Dates": """
        SELECT COUNT(*) AS missing
        FROM chrily.fact_orders f
        LEFT JOIN chrily.dim_date d
        ON f.date_key = d.date_key
        WHERE d.date_key IS NULL;
    """
}

for title, sql in checks.items():
    print(f"\n{title}")
    display(pd.read_sql(sql, engine))


Missing Customers


,missing
0,0



Missing Products


,missing
0,0



Missing Sellers


,missing
0,0



Missing Dates


,missing
0,0


In [40]:
execute_sql_file(
    SQL_PATH / "05_indexes.sql"
)

✓ 05_indexes.sql executed


In [43]:
execute_sql_file(SQL_PATH / "06_business_analysis.sql")

✓ 06_business_analysis.sql executed


In [19]:
execute_sql_file(
    SQL_PATH / "07_views.sql"
)

✓ 07_views.sql executed
